# ④ 発展デモ：グループ分け・次元削減・回帰予測・因果推論
**岡山県 高校教員向け データサイエンス・ハンズオン（発展／デモ）**

このノートは **上から順に実行するだけ** で、機械学習の4つのテーマを「見て確認」できます。
（手を動かす演習ではなく、講師のデモ／自習の確認用です）
1. **クラスタリング（K-means）**：似たもの同士をグループに分ける
2. **主成分分析（PCA）**：たくさんの変数を少ない軸に要約して見える化する
3. **回帰分析と予測（重回帰）**：複数の要因から結果を予測する
4. **因果推論の入り口**：「相関」と「因果」を区別し、交絡を調整する

> 使うライブラリは `scikit-learn`（Colabに最初から入っています）。


## 0. 準備（実行）

In [ ]:
# === データ読み込みの準備（このセルを最初に実行）===
import pandas as pd, io, os
BASE_URL = "https://raw.githubusercontent.com/yasuyuki-nogami/ds-handson/main/data/"

def load(name):
    if os.path.exists("data/" + name):
        return pd.read_csv("data/" + name)
    if BASE_URL:
        try:
            return pd.read_csv(BASE_URL + name)
        except Exception:
            pass
    from google.colab import files
    print(name, " をアップロードしてください")
    up = files.upload(); key = list(up.keys())[0]
    return pd.read_csv(io.BytesIO(up[key]))
print("準備OK")


In [ ]:
!pip -q install japanize-matplotlib
import matplotlib.pyplot as plt
import japanize_matplotlib
import numpy as np
print("準備OK")


---
# 1. クラスタリング（K-means）：47都道府県をグループ分け
「人口・面積・人口密度」の似ている都道府県を、コンピュータが自動でグループ分けします。
正解ラベルを与えずにデータの構造を見つけるので **教師なし学習** と呼びます。


In [ ]:
pref = load("prefecture.csv")

# 分析に使う3つの特徴量（数値の列）を選ぶ
X = pref[["population_2020", "area_km2", "density_per_km2"]]

# --- なぜ「標準化」するのか？ -------------------------------------------
# 人口は数百万、面積は数千、密度は数百…と桁がバラバラ。
# そのまま距離を測ると「数字が大きい列（人口）」ばかりが効いてしまう。
# 標準化＝各列を「平均0・ばらつき1」にそろえる作業。これで公平に比較できる。
# ----------------------------------------------------------------------
from sklearn.preprocessing import StandardScaler
Xs = StandardScaler().fit_transform(X)
print("標準化後の形（行=都道府県, 列=特徴量）:", Xs.shape)


In [ ]:
from sklearn.cluster import KMeans

# n_clusters=3 → 3つのグループに分ける、という指定
# random_state=0 は「毎回同じ結果」にするためのおまじない（再現性のため）
km = KMeans(n_clusters=3, n_init=10, random_state=0)
pref["cluster"] = km.fit_predict(Xs)   # 各都道府県に 0/1/2 のグループ番号が付く

print("各グループの県数:")
print(pref["cluster"].value_counts().sort_index())
pref[["prefecture", "population_2020", "density_per_km2", "cluster"]].head(10)


In [ ]:
# グループごとの平均を見ると、各グループの「性格」が読み取れる
# （例：人口も密度も大きい＝大都市圏、面積が広く密度が低い＝地方 など）
pref.groupby("cluster")[["population_2020","area_km2","density_per_km2"]].mean().round(0)


In [ ]:
# 散布図で色分け（人口 × 人口密度）：同じ色＝同じグループ
plt.figure(figsize=(7,5))
for c in sorted(pref["cluster"].unique()):
    sub = pref[pref["cluster"] == c]
    plt.scatter(sub["population_2020"]/10000, sub["density_per_km2"], label=f"グループ{c}")
plt.xlabel("人口（万人）"); plt.ylabel("人口密度（人/km²）")
plt.title("K-meansによる都道府県のグループ分け")
plt.legend(); plt.show()


In [ ]:
# 各グループにどの都道府県が入ったか
for c in sorted(pref["cluster"].unique()):
    names = pref.loc[pref["cluster"]==c, "prefecture"].tolist()
    print(f"■ グループ{c}（{len(names)}件）:", "、".join(names))


**学びのポイント**
- クラスタリングは「正解ラベルなし」でデータの構造（似た者どうし）を見つける手法。
- 距離の計算では**標準化**が超重要（単位の違いに引っ張られないため）。
- グループ数（`n_clusters`）は人間が決める。目的に応じて2〜4などを試すのが普通。


---
# 2. 主成分分析（PCA）：たくさんの変数を「2つの軸」に要約する
上のクラスタリングは3つの変数を使いました。でも変数が increased（5個・10個…）になると、
散布図では見きれません。そこで **PCA（主成分分析／Principal Component Analysis）** の出番です。

**PCAとは**：たくさんの変数の情報を、なるべく失わずに **少ない数の新しい軸（主成分）** にまとめ直す
「次元削減」の手法です。3次元→2次元にすれば、1枚の散布図で全体を見渡せます。


In [ ]:
from sklearn.decomposition import PCA

# 3つの特徴量（標準化済みの Xs）を、2つの主成分（PC1, PC2）に圧縮する
pca = PCA(n_components=2)
pcs = pca.fit_transform(Xs)   # 各都道府県が (PC1, PC2) の2つの数で表される

# --- 寄与率（explained_variance_ratio_）とは？ -------------------------
# 「元のデータのばらつき（情報量）を、その主成分がどれだけ説明できているか」の割合。
# 例）PC1=0.6 なら、PC1だけで全情報の60%を説明できている、という意味。
# 累積が高い（例：2軸で0.9超）ほど、2次元に落としても情報の損失が少ない。
# ----------------------------------------------------------------------
print("各主成分の寄与率:", pca.explained_variance_ratio_.round(3))
print("累積寄与率(PC1+PC2):", pca.explained_variance_ratio_.cumsum().round(3)[-1])


In [ ]:
# --- 主成分の「意味」を読む：loadings（各元変数の効き方）--------------
# 主成分は元の変数の重み付き合計。重み(loadings)を見ると各軸の意味が分かる。
loadings = pd.DataFrame(
    pca.components_.T,
    index=["人口", "面積", "人口密度"],
    columns=["PC1", "PC2"]
)
print("各主成分への元変数の寄与（loadings）:")
print(loadings.round(2))
print("\n→ 絶対値が大きい変数ほど、その主成分の意味を強く決めている")


In [ ]:
# PCAで得た2軸(PC1, PC2)で散布図。色はK-meansのグループ。
# 3変数の情報を1枚の図に「要約」して見られるのがPCAの利点。
plt.figure(figsize=(7,6))
for c in sorted(pref["cluster"].unique()):
    m = pref["cluster"] == c
    plt.scatter(pcs[m, 0], pcs[m, 1], label=f"グループ{c}")

# 特徴的な都道府県だけ名前を表示（図が名前だらけにならないように）
for name in ["東京都", "北海道", "大阪府", "神奈川県", "鳥取県"]:
    if name in pref["prefecture"].values:
        i = pref.index[pref["prefecture"] == name][0]
        plt.annotate(name, (pcs[i, 0], pcs[i, 1]))

plt.xlabel("第1主成分 PC1"); plt.ylabel("第2主成分 PC2")
plt.title("PCAで2次元に要約した47都道府県（色=K-meansのグループ）")
plt.legend(); plt.tight_layout(); plt.show()


**学びのポイント**
- PCAは **次元削減**：多くの変数を、情報をできるだけ保ったまま少ない軸に要約する。
- **寄与率**で「その軸がどれだけ情報を持つか」、**loadings**で「その軸が何を表すか」を読む。
- K-means（分ける）と PCA（見える化する）は相性がよく、**分けた結果を2次元で確認**できる。
- 実データでは変数が多いほどPCAの価値が上がる（今回は3変数の入門例）。


---
# 3. 回帰分析と予測（重回帰）
複数の要因から「テストの点数」を予測するモデルを作ります。
※ `class_sample.csv` は練習用の**架空データ**です。


In [ ]:
df = load("class_sample.csv")

# 説明変数（原因になりそうな要因）と、目的変数（予測したいもの）を決める
features = ["study_min", "sleep_hours", "breakfast", "smartphone_hours"]
X = df[features]
y = df["test_score"]

# --- なぜ学習用とテスト用に分ける？ ------------------------------------
# 同じデータで学習・採点すると「丸暗記」でも高得点に見えてしまう。
# 一部を「テスト用」に取り分け、学習に使っていないデータで実力を測る。
# ----------------------------------------------------------------------
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
print("学習用", X_train.shape, " テスト用", X_test.shape)


In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression().fit(X_train, y_train)

# 係数 = 「その要因が1単位増えると点数が何点変わるか」（プラスは上げる/マイナスは下げる要因）
coef = pd.Series(model.coef_, index=features).round(2)
print("係数（1単位あたり点数がどれだけ変わるか）")
print(coef)
print("切片:", round(model.intercept_, 1))


In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error
pred = model.predict(X_test)
# R^2: 1に近いほど当てはまり良い / MAE: 予測が平均何点ずれるか
print("R^2（1に近いほど良い）:", round(r2_score(y_test, pred), 3))
print("平均絶対誤差（点）:", round(mean_absolute_error(y_test, pred), 2))

# 予測 vs 実際 を散布図で（点が対角線に近いほど良い予測）
plt.figure(figsize=(5,5))
plt.scatter(y_test, pred, alpha=0.4)
plt.plot([0,100],[0,100], "r--")
plt.xlabel("実際の点数"); plt.ylabel("予測した点数"); plt.title("予測の精度")
plt.show()


In [ ]:
# 学習したモデルで、新しい生徒の点数を予測してみる
new_student = pd.DataFrame([{
    "study_min": 90, "sleep_hours": 7.0, "breakfast": 1, "smartphone_hours": 1.5
}])
print("この生徒の予測点数:", round(model.predict(new_student)[0], 1))


**学びのポイント**
- 重回帰は「複数の要因」から結果を予測し、各要因の効き方（係数）も分かる。
- **学習用/テスト用に分ける**ことで、予測の"本当の実力"を測れる。
- 係数はあくまで「このデータ上の関係」。因果とは限らない（→ 次の章へ）。


---
# 4. 因果推論の入り口：「相関」と「因果」は違う
**問い**：「塾に通うと成績は上がる？」

観察データをそのまま比べると間違えることがあります。理由は **交絡（こうらく）**。
ここでは、答えが分かっている**シミュレーションデータ**で確かめます。

今回の"真実"の設定：
- 家庭の教育投資 `Z` が、**塾に通うか `X`** と **成績 `Y`** の両方に影響する（＝交絡因子）
- 塾の**本当の効果は +3点**（この値を当てられるか見ます）


In [ ]:
# データを人工的に作る（"正解"が分かっている状態で試すため）
rng = np.random.default_rng(0)
n = 2000
Z = rng.normal(0, 1, n)                       # 家庭の教育投資（交絡因子）
p = 1 / (1 + np.exp(-(1.5*Z)))                # Zが高いほど塾に通いやすい
X = rng.binomial(1, p)                         # 塾に通う=1, 通わない=0
true_effect = 3.0                              # ← 塾の"本当の"効果（+3点）
Y = 50 + true_effect*X + 8*Z + rng.normal(0, 5, n)   # 成績（Zの影響が大きい点に注目）

sim = pd.DataFrame({"教育投資Z": Z, "塾X": X, "成績Y": Y})
sim.head()


In [ ]:
# ① そのまま比較（交絡を無視）：塾に通う人 vs 通わない人の平均点
naive = sim.groupby("塾X")["成績Y"].mean()
print(naive)
print("素朴な差（見かけの効果）:", round(naive[1] - naive[0], 2), "点")
print("→ 本当の効果 +3点 より大きく出てしまう（Zのせいで水増しされる）")


In [ ]:
# ② 交絡因子Zを"調整"して比較（重回帰にZも入れる）
m = LinearRegression().fit(sim[["塾X", "教育投資Z"]], sim["成績Y"])
print("Zを調整した塾Xの効果:", round(m.coef_[0], 2), "点")
print("→ 本当の効果 +3点 に近づく！ これが「交絡の調整」")


### まとめ：因果を語るときの注意（学びのポイント）
- **相関があっても因果とは限らない**（交絡などのリスク）。
- 観察データでは、**交絡因子を測って調整**することが第一歩（今回は重回帰でZを調整）。
- 最も強い方法は **ランダム化実験（RCT）**：Xを"くじ引き"で割り当てれば、Zの偏りが消える。
- 現実には測れない交絡もある → 「言えること／言えないこと」を丁寧に区別するのがデータリテラシー。

> 生徒への問いかけ例：「このデータだけで"塾に行けば必ず伸びる"と言い切れる？ 何が足りない？」
